In [1]:
 from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(
    temperature=0,
    groq_api_key='enter api key',
    model_name="openai/gpt-oss-120b"
)
response = llm.invoke("The first person to land on moon was ...")
print(response.content)

The first person to set foot on the Moon was **Neil Armstrong**, who did so on July 20 1969 during NASA’s Apollo 11 mission. He famously described the moment as “one small step for [a] man, one giant leap for mankind.”


In [3]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/principal-india-strategy-business-development-india/job/R-83424")
page_data = loader.load().pop().page_content
print(page_data)

USER_AGENT environment variable not set, consider setting it to identify your requests.






















Principal India Strategy & Business Development, India












































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu


In [4]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):
        """
)

chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [5]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Principal India Strategy & Business Development, India',
 'experience': 'Minimum 12 years of relevant strategy, business development or management consultancy experience, preferably managing rapid-growth, consumer-facing digital and/or retail businesses.',
 'skills': ['Strategic leadership',
  'Enterprise-wide strategy development and execution',
  'Project management',
  'Cross-functional collaboration',
  'Stakeholder management',
  'Retail experience and business model expertise',
  'Digital and physical commerce integration',
  'Strong business judgment',
  'Analytical and research capabilities',
  'Ability to translate strategy into execution',
  'Growth-oriented mindset',
  'Passion for sport and fashion'],
 'description': 'Nike India is seeking a Principal, India Strategy & Business Development based in Bengaluru. Reporting to the General Manager, India and part of the Nike India Leadership Team, the role will develop and translate business strategy into a clear busine

In [6]:
type(json_res)

dict

In [9]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [10]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [13]:
job = json_res
job['skills']

['Strategic leadership',
 'Enterprise-wide strategy development and execution',
 'Project management',
 'Cross-functional collaboration',
 'Stakeholder management',
 'Retail experience and business model expertise',
 'Digital and physical commerce integration',
 'Strong business judgment',
 'Analytical and research capabilities',
 'Ability to translate strategy into execution',
 'Growth-oriented mindset',
 'Passion for sport and fashion']

In [14]:
links = collection.query(query_texts=job['skills'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/kotlin-backend-portfolio'}],
 [{'links': 'https://example.com/kotlin-backend-portfolio'},
  {'links': 'https://example.com/java-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/wordpress-portfolio'}],
 [{'links': 'https://example.com/xamarin-portfolio'},
  {'links': 'https://example.com/typescript-frontend-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/react-portfolio'}],
 [{'links': 'https://example.com/vue-portfolio'},
  {'links': 'https://example.com/python-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/xamarin-portfolio'}],
 [{'links': 'https://example.com/angular-portfolio'},
  {'links': 'https://example.com/vue-portfolio'}],
 [{'links': 'https://example.com/angular-portfolio'},
  {'links': 'https://example.com/typescript-frontend-portfolio'}

In [15]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}

        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools.
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability,
        process optimization, cost reduction, and heightened overall efficiency.
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ.
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):

        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Empowering Nike India’s Retail & Digital Growth with AtliQ’s AI‑Driven Solutions  

Hi [Hiring Manager’s Name],

I’m Mohan, Business Development Executive at AtliQ, an AI‑powered software consulting firm that partners with fast‑growing consumer brands to turn strategic vision into scalable, technology‑enabled reality.

**Why AtliQ aligns with Nike India’s objectives**

- **End‑to‑end automation & DevOps excellence** – Our DevOps practice accelerates release cycles, ensures platform stability, and reduces operational costs, enabling rapid rollout of new retail concepts across both digital and physical stores.  
- **Robust backend platforms** – Leveraging Kotlin and Java, we build high‑performance, secure services that power omnichannel experiences, from mobile apps to in‑store kiosks.  
- **E‑commerce & marketplace expertise** – Our Magento implementations deliver seamless product catalogs, personalized merchandising, and integrated payment flows—critical for scaling Nike’s dir